# 02 · Dates & Times: `datetime`, `time`, `zoneinfo`

**Module 12 – Standard Library Essentials** · Estimated time: 45 min

> Expands 📗 *Datetime Module* (time, date, arithmetic) with formatting/parsing, ISO 8601, **time zones** (`zoneinfo`, 3.9+), and current deprecations.

### 🎯 What you will learn
- `date`, `time`, `datetime`, `timedelta`
- Date arithmetic
- Formatting with `strftime` and parsing with `strptime`; ISO 8601
- Naive vs **aware** datetimes; `zoneinfo` time zones
- ⚠️ `utcnow()`/`utcfromtimestamp()` deprecations
- The `time` module: timestamps, `sleep`, `perf_counter`
- `calendar`

## 1. `time` objects (📗)

In [1]:
import datetime

t = datetime.time(4, 20, 1)
print(t)
print("hour  :", t.hour)
print("minute:", t.minute)
print("second:", t.second)
print("microsecond:", t.microsecond)
print("tzinfo:", t.tzinfo)
print("Earliest  :", datetime.time.min)
print("Latest    :", datetime.time.max)
print("Resolution:", datetime.time.resolution)

04:20:01
hour  : 4
minute: 20
second: 1
microsecond: 0
tzinfo: None
Earliest  : 00:00:00
Latest    : 23:59:59.999999
Resolution: 0:00:00.000001


## 2. `date` objects (📗)

In [2]:
today = datetime.date.today()
print(today)
print("ctime:", today.ctime())
print("tuple:", today.timetuple()[:3])
print("ordinal:", today.toordinal())
print("Year:", today.year, "| Month:", today.month, "| Day:", today.day)
print("Weekday (Mon=0):", today.weekday(), "| ISO weekday (Mon=1):", today.isoweekday())
print("Earliest:", datetime.date.min, "| Latest:", datetime.date.max)

2026-10-20
ctime: Tue Oct 20 00:00:00 2026
tuple: (2026, 10, 20)
ordinal: 739909
Year: 2026 | Month: 10 | Day: 20
Weekday (Mon=0): 1 | ISO weekday (Mon=1): 2
Earliest: 0001-01-01 | Latest: 9999-12-31


In [3]:
d1 = datetime.date(2015, 3, 11)
print("d1:", d1)
d2 = d1.replace(year=1990)          # dates are immutable → replace() returns a new one
print("d2:", d2)

d1: 2015-03-11
d2: 1990-03-11


## 3. Date arithmetic with `timedelta` (📗)

In [4]:
print(d1 - d2)                       # a timedelta
diff = d1 - d2
print(diff.days, "days")
print(datetime.date(2026, 10, 20) + datetime.timedelta(days=100))
print(datetime.date(2026, 12, 25) - datetime.date(2026, 10, 20))

week = datetime.timedelta(weeks=1, hours=3, minutes=30)
print(week, week.total_seconds())

9131 days, 0:00:00
9131 days
2027-01-28
66 days, 0:00:00
7 days, 3:30:00 617400.0


In [5]:
# Age calculator
from datetime import date
def age(birthday, on=None):
    on = on or date.today()
    return on.year - birthday.year - ((on.month, on.day) < (birthday.month, birthday.day))

print(age(date(2001, 7, 19), on=date(2026, 10, 20)))

25


> 💡 `timedelta` has no "months" or "years" (they have variable length). Use `date.replace()` or the third-party `python-dateutil` (`relativedelta`) for calendar maths.

## 4. `datetime` = date + time

In [6]:
from datetime import datetime, timedelta

now = datetime.now()
print(now)
meeting = datetime(2026, 10, 20, 14, 30)
print(meeting, meeting.date(), meeting.time())
print(meeting + timedelta(hours=1, minutes=15))
print(datetime.combine(date(2026, 1, 1), datetime.min.time()))

2026-10-20 19:44:09.480918
2026-10-20 14:30:00 2026-10-20 14:30:00
2026-10-20 15:45:00
2026-01-01 00:00:00


## 5. Formatting (`strftime`) and parsing (`strptime`)

| Code | Meaning | Example |
|---|---|---|
| `%Y` / `%y` | year | `2026` / `26` |
| `%m` | month (01–12) | `10` |
| `%B` / `%b` | month name | `October` / `Oct` |
| `%d` | day of month | `20` |
| `%A` / `%a` | weekday | `Tuesday` / `Tue` |
| `%H` / `%I` | hour 24h / 12h | `14` / `02` |
| `%M`, `%S` | minute, second | `30`, `05` |
| `%p` | AM/PM | `PM` |
| `%j` | day of year | `293` |
| `%Z`, `%z` | timezone name / offset | `IST`, `+0530` |

In [7]:
print(meeting.strftime("%d %B %Y, %I:%M %p"))
print(meeting.strftime("%A %d/%m/%y"))
print(f"{meeting:%Y-%m-%d %H:%M}")                  # format spec inside f-strings

parsed = datetime.strptime("20/10/2026 18:45", "%d/%m/%Y %H:%M")
print(parsed, type(parsed))
print(date.strptime("2026-10-20", "%Y-%m-%d"))     # 🆕 3.14: date.strptime / time.strptime

20 October 2026, 02:30 PM
Tuesday 20/10/26
2026-10-20 14:30
2026-10-20 18:45:00 <class 'datetime.datetime'>
2026-10-20


> 🆕 **Python 3.15:** parsing a day-of-month (`%d`) **without a year** now raises `ValueError` (it was ambiguous: Feb 29 can't be parsed without a year):

In [8]:
try:
    datetime.strptime("20 Oct", "%d %b")
except ValueError as e:
    print("ValueError:", str(e)[:80], "...")

ValueError: Day of month directive '%d' may not be used without a year directive. Parsing da ...


### ISO 8601: the format to use for data exchange

In [9]:
iso = meeting.isoformat()
print(iso)
print(datetime.fromisoformat("2026-10-20T14:30:00+05:30"))   # 3.11+ accepts most ISO 8601 forms
print(date.fromisoformat("2026-10-20"))

2026-10-20T14:30:00
2026-10-20 14:30:00+05:30
2026-10-20


## 6. Time zones: naive vs aware

- **Naive** datetime: no time zone attached (`tzinfo=None`), ambiguous!
- **Aware** datetime: knows its zone/offset; safe for comparisons across zones

✅ **Best practice:** store and compute in **UTC**, convert to local time only for display.

In [10]:
from datetime import UTC                       # 3.11+ alias for timezone.utc
from zoneinfo import ZoneInfo                  # 3.9+: the IANA time-zone database

now_utc = datetime.now(UTC)                    # ✅ aware "now" in UTC
print(now_utc, now_utc.tzinfo)

launch = datetime(2026, 10, 20, 9, 0, tzinfo=ZoneInfo("Asia/Kolkata"))
print("Kolkata :", launch)
for tz in ["UTC", "Europe/London", "America/New_York", "Asia/Tokyo"]:
    print(f"{tz:<17}", launch.astimezone(ZoneInfo(tz)).strftime("%Y-%m-%d %H:%M %Z (%z)"))

2026-10-20 14:14:09.495210+00:00 UTC
Kolkata : 2026-10-20 09:00:00+05:30
UTC               2026-10-20 03:30 UTC (+0000)
Europe/London     2026-10-20 04:30 BST (+0100)
America/New_York  2026-10-19 23:30 EDT (-0400)
Asia/Tokyo        2026-10-20 12:30 JST (+0900)


In [11]:
# Daylight-saving time is handled automatically
ny = ZoneInfo("America/New_York")
print(datetime(2026, 1, 15, 12, tzinfo=ny).utcoffset())   # winter: -5h
print(datetime(2026, 7, 15, 12, tzinfo=ny).utcoffset())   # summer: -4h

-1 day, 19:00:00
-1 day, 20:00:00


In [12]:
datetime(2026, 1, 1) < datetime(2026, 1, 1, tzinfo=UTC)   # ❌ TypeError: can't compare naive and aware

TypeError: can't compare offset-naive and offset-aware datetimes

> ⚠️ **Deprecated (3.12):** `datetime.utcnow()` and `datetime.utcfromtimestamp()`. They return *naive* datetimes that look like local time, a common bug source. Use `datetime.now(UTC)` and `datetime.fromtimestamp(ts, UTC)`.
>
> ⚠️ **Old:** the third-party `pytz` library. Use the built-in `zoneinfo` (on Windows you may need `pip install tzdata`).

## 7. Timestamps & the `time` module

In [13]:
import time

ts = time.time()                         # seconds since 1 Jan 1970 UTC (the Unix epoch)
print(ts)
print(datetime.fromtimestamp(ts, UTC))
print(int(datetime(2026, 1, 1, tzinfo=UTC).timestamp()))

start = time.perf_counter()              # ✅ for measuring durations
time.sleep(0.2)                          # pause execution
print(f"slept {time.perf_counter() - start:.3f}s")
print(time.strftime("%H:%M:%S"))

1792505649.543644
2026-10-20 14:14:09.543644+00:00
1767225600


slept 0.205s
19:44:09


## 8. `calendar`

In [14]:
import calendar
print(calendar.month(2026, 10))
print(calendar.isleap(2028), calendar.monthrange(2026, 2))   # (weekday of 1st, number of days)
print(list(calendar.day_name))

    October 2026
Mo Tu We Th Fr Sa Su
          1  2  3  4
 5  6  7  8  9 10 11
12 13 14 15 16 17 18
19 20 21 22 23 24 25
26 27 28 29 30 31

True (calendar.SUNDAY, 28)
['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


## 📝 Summary
- `date`, `time`, `datetime`, `timedelta`; objects are immutable (`replace()` returns new ones).
- `strftime` formats, `strptime` parses; prefer ISO 8601 (`isoformat`/`fromisoformat`) for data.
- ✅ Use **aware** datetimes: `datetime.now(UTC)` + `zoneinfo.ZoneInfo`.
- ⚠️ `utcnow()` is deprecated; `pytz` is legacy.
- `time.perf_counter()` for durations, `time.sleep()` to pause.

## 🏋️ Practice
1. How many days until your next birthday?
2. Print today's date as `Tuesday, 20 October 2026`.
3. Parse `"2026-12-31 23:59"` and add 2 minutes. What's the new year?
4. A meeting is at 10:00 in London on 3 Nov 2026. What time is it in Mumbai and San Francisco?
5. List all Fridays the 13th in 2026.

<details><summary>Show solutions</summary>

```python
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

today = date.today()
bday = date(today.year, 7, 19)
if bday < today:
    bday = bday.replace(year=today.year + 1)
print((bday - today).days)

print(today.strftime("%A, %d %B %Y"))

print((datetime.strptime("2026-12-31 23:59", "%Y-%m-%d %H:%M") + timedelta(minutes=2)).year)

m = datetime(2026, 11, 3, 10, tzinfo=ZoneInfo("Europe/London"))
print(m.astimezone(ZoneInfo("Asia/Kolkata")), m.astimezone(ZoneInfo("America/Los_Angeles")))

print([date(2026, mo, 13) for mo in range(1, 13) if date(2026, mo, 13).weekday() == 4])
```
</details>